# Rec Centre Bookings: Inspect, Clean, Transform
SDEV2501 · Days 10-11 · **Student notebook**

The City's recreation department exported every booking from **September 2026** and handed it to us. We want two things out of it: which facility makes the most money, and when people actually show up.

We're not getting either one yet. Whoever entered this data had a fresh idea about how to spell "Lane Swim" every time they sat down. So first we work out *how* messy it is, then we clean it, then we reshape it into something that can answer a question without lying to us.

Keep the exercise guide (`bookings_wrangling_exercises_v2.md`) open beside this. Every **Your turn** has an empty code cell under it. That one's yours.

**Files (same folder as this notebook):** `bookings_sept_messy.csv`, `facilities.csv`

> Mock data. The bookings and members are made up; the neighbourhoods are real Edmonton ones.

---
# Part 1: Inspect & Clean (Day 10)
One rule for today: **look before you touch.** It's tempting to fix the first problem you spot. Don't. Find everything first and write it down, *then* start changing things. Fix as you go and you'll tidy up one problem while quietly burying two others.

## 1. Load
**Your turn:** load `bookings_sept_messy.csv` into a DataFrame called `df` and show the first 10 rows.

**Your turn:** how many rows and columns are there, and what are the columns called?

## 2. Inspect
Nothing in this section changes `df`. We're detectives right now, not janitors. Every time you find a problem, add it to the **findings table** at the end of section 2.

### 2.1 Missing values
**Your turn:** count the missing values in each column.

### 2.2 Duplicates
**Your turn:** how many rows are exact copies of an earlier row? Show every copy, sorted by booking ID so the pairs sit together.

**Your turn:** every booking should have its own ID. Check whether any `Booking ID` appears more than once.

### 2.3 Unexpected categories
**Your turn:** list every value in `Activity`, `Status` and `Facility ID`, with counts. To you, `Lane Swim` and `lane swim` are the same activity. To pandas they're strangers. How many strangers are there?

### 2.4 Impossible values
An **impossible** value can't be true no matter what. A booking can't last less than no time, and September doesn't get a 31st just because someone typed one.

**Your turn:** `Hours ` is stored as text (more on that in 2.5), so make a temporary numeric copy called `hours_check`, then look at its summary statistics. Then show any rows with zero or negative hours.

### 2.5 Wrong data types
**Your turn:** check the data type of every column. Which ones look wrong?

**Your turn:** find the `Hours ` values that are **not blank** but **didn't convert** to a number. Then list the distinct values in `Hourly Rate`.

### 2.6 Suspicious distributions
A **suspicious** value is possible, but weird enough that a human should look at it before anyone trusts it. Impossible means *wrong*. Suspicious means *go check*.

**Your turn:** plot a histogram of `hours_check`. Then make a numeric copy of the rate called `rate_check` (remove the `$` first) and plot that too. What sits far away from everything else?

### Findings
**Your turn:** fill this in as you go, one row per problem. Double-click the cell to edit it.

| # | Problem | Column(s) | How you found it |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |
| 5 | | | |
| 6 | | | |
| 7 | | | |
| 8 | | | |
| 9 | | | |

## 3. Clean
Now we get to touch things. Work down the findings table, and for every fix ask one question: **do I know the right answer, or am I guessing?** If you know, fix it. If you'd be guessing, flag it or leave it.

Guessing and calling it "cleaning" is how wrong numbers end up in a report with your name on it.

### 3.1 Rename columns
**Your turn:** clean every column name in one go: strip spaces, lowercase, spaces to underscores. Then rename `member_#` to `member_id`.

### 3.2 Remove duplicates
**Your turn:** drop the exact duplicates and print how many rows were removed.

**Your turn:** remove the 20-hour copy of booking 5071, then confirm every `booking_id` is unique.

### 3.3 Fix categories
**Your turn:** tidy `activity`: strip spaces, collapse double spaces to one, then Title Case. Check the counts.

**Your turn:** the rules got most of them. What's left needs a human, because no string method knows that `Swim` means `Lane Swim`. Use a dictionary to turn `Swim`, `Skate` and `Gym Drop In` into the proper names.

**Your turn:** clean `status` too. Do it with a `for` loop first, then the vectorized way (one instruction for the whole column, no loop). Writing it the long way once is the best argument for never doing it again.

**Your turn:** make every `facility_id` uppercase.

### 3.4 Fix data types
**Your turn:** convert `hours` to numbers (anything that won't convert becomes `NaN`). Remove the `$` from `hourly_rate` and convert it too. Check the dtypes.

### 3.5 Missing values
You get three options for a blank: **drop** the row, **fill** it in, or **leave** it. The choice comes down to one question: do you actually know what belongs there?

**Your turn:** for rates, we do know. Every activity has a standard hourly rate. Put the rates in a dictionary and fill **only** the missing `hourly_rate` values from it.

**Your turn:** hours are different. We have no idea how long those bookings were, so the best we can do is an educated guess, and a guess should be labelled as one. Add a True/False column `hours_filled` that marks them, then fill them with the median.

### 3.6 Impossible and suspicious values
**Your turn:** before you run anything, decide: fix, flag, or leave? Write one line of reasoning for each (double-click this cell to edit it). "It felt right" doesn't count. If you're working with a partner, see if you agree.

- Hours of -2
- A 24-hour court rental
- A $450/hour court rate

Then: add a True/False `needs_review` column for the hours problems, set the impossible hours to `NaN` so they don't count in totals, and fix the rate.

### 3.7 Save
Before saving, look back at the findings table. Did every problem get fixed, flagged, or left on purpose? (The dates are still a disaster. That's next class's problem.)

**Your turn:** save the cleaned data to `bookings_cleaned.csv`, without the index.

---
# Part 2: Transform & Reshape (Day 11)
New day, clean slate, and (if Part 1 went well) a clean CSV. Restart the kernel, run the imports cell at the top, then start here.

**Your turn:** load `bookings_cleaned.csv` into `df` and check the data types.

## 4. Select
Before you can do anything clever with a table, you have to grab the bit you want.

**Your turn:** select the `activity` column. Then select `booking_id`, `activity` and `hours` together. Show the first 5 rows of each.

**Your turn:** use `.loc` to get rows 0 to 4 of `booking_id` and `activity` only.

## 5. Filter
Filtering asks the table a yes/no question about every row and keeps the yeses.

**Your turn:** show only the court rentals.

**Your turn:** show completed court rentals (two conditions). Then count the bookings that are either Lane Swim or Public Skate. Then count the bookings that are **not** flagged for review.

## 6. Create and modify columns
**Your turn:** create `booking_total`, the cost of each booking (hours × hourly rate). Also create a True/False `is_long` column for bookings over 2 hours.

**Your turn:** cancelled bookings and no-shows didn't earn the City a cent. Set their `booking_total` to 0.

## 7. Dates and data types
**Your turn:** convert `booking_date` to real dates. It's in several formats, because of course it is. Before saving the result, look at the rows that didn't convert.

**Your turn:** save the converted dates to the column, then check the earliest and latest date. Flag any booking with a missing date or a date after September.

**Your turn:** pull the day of the week out of each date into a new column. Then convert `activity`, `status` and `facility_id` to the `category` type and check the dtypes.

## 8. Normalize
Heads up: *normalize* here has nothing to do with database normalization. Same word, different job. Here it means putting numbers on a common scale so you can compare them.

- **Min-max scaling** squeezes values into 0 to 1: `(x - min) / (max - min)`
- **Z-score** is how many standard deviations a value is from the mean: `(x - mean) / std`

**Your turn:** scale `booking_total` both ways and compare them with `.describe()`.

## 9. Merge
The bookings only have a facility code. The names and neighbourhoods live in `facilities.csv`. A **merge** adds columns from one table to another using a column they share (the **key**). If you've used VLOOKUP in Excel, it's that, minus dragging a formula down 140 rows.

**Your turn:** load `facilities.csv`. Then left-merge it onto the bookings using `facility_id`, with `indicator=True`, and count how many rows matched.

**Your turn:** show the bookings that didn't find a facility.

## 10. Group and summarize
Back to the question we started with, now that the data won't lie to us.

**Your turn:** which facility made the most money? Use completed bookings that aren't flagged for review. Show the number of bookings and the total for each facility, biggest total first.

**Your turn:** which day of the week has the most completed bookings?

---
## Wrap-up
Write two reasons why wrangling has to happen *before* analysis. Use something you saw in this notebook as evidence for each.

1.
2.